# 1. Identity Fundamentals

## Identity is the new security perimeter

In the old world, your corporate network *was* the security boundary — everyone inside the firewall was trusted. That model is dead. People work from home, apps are in the cloud, and data flows everywhere.

**Identity** has replaced the network perimeter. Every request must prove *who* is asking (authentication) and *what* they're allowed to do (authorization).

## Setup

```bash
cd security-certs/sc-900/02-identity-and-entra
docker compose up -d
uv sync
```

Select the `SC-900 (Python)` kernel.

---
## Authentication vs Authorization

These two get confused constantly. Here's the difference:

| | Authentication (AuthN) | Authorization (AuthZ) |
|-|----------------------|---------------------|
| **Question** | *Who are you?* | *What can you do?* |
| **Happens** | First | Second (after authn) |
| **Proof** | Password, MFA, biometric, certificate | Roles, permissions, policies |
| **Failure** | 401 Unauthorized | 403 Forbidden |
| **Azure service** | Entra ID | RBAC, Conditional Access |

**Analogy**: authentication is showing your passport at the airport. Authorization is whether your boarding pass lets you into first class.

### One more framing you may meet: the four pillars of identity

The current SC-900 study guide phrases this domain as *authentication / authorization /
identity providers / directory services / federation*. Older Microsoft courseware describes
the same ground as **four pillars of an identity infrastructure**. If a question uses the
pillar vocabulary, this is it:

| Pillar | Question it answers | Where it shows up in Entra |
|--------|---------------------|----------------------------|
| **Administration** | Who creates, updates, deletes and governs identities? | User/group management, entitlement management, lifecycle workflows |
| **Authentication** | How much proof does the system need that you are you? | Password + MFA, passkeys, Windows Hello |
| **Authorization** | What is this identity allowed to do? | Entra roles, Azure RBAC, Conditional Access grants |
| **Auditing** | Who did what, when — and can we prove it? | Sign-in logs, audit logs, access reviews |

Every capability in the rest of this lab hangs off one of those four.

In [ ]:
import httpx, json, base64

ENTRA = 'http://localhost:9100/contoso'
TOKEN_URL = f'{ENTRA}/oauth2/v2.0/token'
API_B = 'http://localhost:8002'

def decode(t):
    p = t.split('.')[1]
    return json.loads(base64.urlsafe_b64decode(p + '=' * (-len(p) % 4)))

def get_token(password='alice-password', scope='api://api-b/Files.Read'):
    return httpx.post(TOKEN_URL, data={
        'grant_type': 'password',
        'client_id': 'api-a-client-id',
        'client_secret': 'api-a-secret-value',
        'username': 'alice@contoso.com',
        'password': password,
        'scope': scope,
    })

# --- AUTHENTICATION: prove who you are ---
print('=== Step 1: Authentication (prove your identity) ===')
r = get_token()
assert r.status_code == 200, f'the IdP should have issued a token, got {r.status_code}'
token = r.json()['access_token']
claims = decode(token)
print(f'✅ Authenticated as: {claims["upn"]}')
print(f'   Token audience: {claims["aud"]}')
print(f'   Scopes granted: {claims["scp"]}')

# --- AUTHORIZATION: check what you can do ---
print('\n=== Step 2: Authorization (can you access /files?) ===')
r = httpx.get(f'{API_B}/files', headers={'Authorization': f'Bearer {token}'})
assert r.status_code == 200, f'Files.Read should be enough to list files, got {r.status_code}'
print(f'Status: {r.status_code}')
print(json.dumps(r.json(), indent=2))

# --- Failed AUTHENTICATION: we cannot prove who we are -> 401 ---
print('\n=== Step 3: Failed authentication (wrong password) → 401 ===')
r = get_token(password='WRONG-password')
assert r.status_code == 401, f'a bad password is an authn failure (401), got {r.status_code}'
print(f'Status: {r.status_code} — {r.json()["detail"]}')

print('\n=== Step 3b: No token at all → also 401, not 403 ===')
r = httpx.get(f'{API_B}/files')
assert r.status_code == 401, f'a missing token is an authn failure (401), got {r.status_code}'
print(f'Status: {r.status_code} — {r.json()["detail"]}')

# --- Failed AUTHORIZATION: identity proven, but the token lacks the scope -> 403 ---
print('\n=== Step 4: Failed authorization (authenticated, wrong scope) → 403 ===')
# (the mock IdP does not police which scopes a client may ask for; real Entra would,
#  but the point here is what the *resource* does when the scope is missing)
weak = get_token(scope='api://api-b/Files.Write').json()['access_token']
print(f'   signed in fine as {decode(weak)["upn"]}, but scp = {decode(weak)["scp"]!r}')
r = httpx.get(f'{API_B}/files', headers={'Authorization': f'Bearer {weak}'})
assert r.status_code == 403, f'known identity + missing scope is an authz failure (403), got {r.status_code}'
print(f'Status: {r.status_code} — {r.json()["detail"]}')

print('\n401 = "I do not know who you are."   403 = "I know exactly who you are, and the answer is no."')

---
## Identity Providers (IdP)

An **identity provider** is the system that creates, maintains, and manages identity information and provides authentication services. Instead of every app managing its own usernames and passwords, you delegate to a central IdP.

| IdP | Who uses it |
|-----|-------------|
| **Microsoft Entra ID** | Azure, Microsoft 365, thousands of SaaS apps |
| Google Identity | Google Workspace, Android |
| Okta | Enterprise apps |
| On-premises AD | Traditional Windows domains |

### How it works (simplified)

```
User → App: "I want to log in"
App → IdP: "Please authenticate this user"
IdP → User: "Enter credentials + MFA"
User → IdP: credentials
IdP → App: signed token (JWT) proving user's identity
App: validates token signature, reads claims, grants access
```

The mock Entra server in this lab IS an identity provider — it issues signed JWTs just like real Entra ID.

In [ ]:
# The OIDC discovery document tells apps where to find the IdP's endpoints
discovery = httpx.get(f'{ENTRA}/v2.0/.well-known/openid-configuration').json()
print('OIDC Discovery Document (how apps find the IdP):\n')
for key in ['issuer', 'token_endpoint', 'jwks_uri']:
    print(f'  {key}: {discovery[key]}')

print('\n📖 In real Entra ID, this URL is:')
print('   https://login.microsoftonline.com/<tenant-id>/v2.0/.well-known/openid-configuration')

---
## Identity protocols — OAuth 2.0, OIDC, SAML

The exam expects you to know the difference. They all deliver tokens from an IdP to an app, but they solve different problems.

| Protocol | Purpose | Token format | Used by |
|----------|---------|--------------|---------|
| **OAuth 2.0** | *Authorization* — delegate API access ("let this app read my calendar") | Access token (often JWT) | Modern APIs, mobile/SPAs |
| **OpenID Connect (OIDC)** | *Authentication* — "who is the user?" — a thin layer **on top of** OAuth 2.0 | ID token (JWT) | Modern web sign-in |
| **SAML 2.0** | Enterprise SSO, older — browser posts signed XML assertions | SAML assertion (XML) | Legacy SaaS, on-prem federation |

### Quick rule of thumb
- **OAuth 2.0 alone** → API authorization.
- **OIDC** = OAuth 2.0 **+** an `id_token` so the app learns *who* the user is.
- **SAML** → older enterprise SSO (think ADFS to Salesforce). Entra ID supports all three.

The mock IdP in this lab speaks **OAuth 2.0 / OIDC** (JSON + JWT over HTTP). A SAML IdP would sign an XML assertion instead, but the *concept* — IdP issues signed proof, app verifies signature — is the same.

---
## Directory services and Active Directory

A **directory service** stores identity objects (users, groups, devices) in a hierarchical structure and provides lookup/authentication services.

| Service | Where | Protocol | Purpose |
|---------|-------|----------|---------|
| **Active Directory Domain Services (AD DS)** | On-premises | LDAP, Kerberos | Traditional Windows domain controller |
| **Microsoft Entra ID** | Cloud | OAuth2, OIDC, SAML | Cloud-native identity for Azure, M365, SaaS |
| **Microsoft Entra Domain Services** | Cloud | LDAP, Kerberos (managed) | Lift-and-shift AD to Azure without managing DCs |

### Exam tip

- AD DS = on-prem, you manage domain controllers
- Entra ID = cloud, no domain controllers, uses modern protocols (OAuth2/OIDC)
- Entra Domain Services = managed LDAP/Kerberos in Azure (for legacy apps)

They are **not** the same thing. AD DS ≠ Entra ID. Entra ID is not "AD in the cloud".

---
## Trusting a token — from bad practice to best practice

A JWT has three base64url parts: `header.payload.signature`. Anyone can read the payload (it's not encrypted). The **signature** is what proves the token really came from the IdP.

Below we compare a naive app that just trusts the payload vs. a correct app that verifies the signature using the IdP's public keys (JWKS).

In [ ]:
# ❌ BAD: trust the token payload without verifying the signature
# Attacker can forge any claims they like — the app will happily believe them.
import json, base64, httpx
from jose import jwt as jose_jwt
from jose.exceptions import JWTError

def bad_who_is_this(token: str) -> dict:
    payload_b64 = token.split('.')[1]
    payload_b64 += '=' * (-len(payload_b64) % 4)
    return json.loads(base64.urlsafe_b64decode(payload_b64))

# Forge a token that *claims* to be from the admin. We never sign it properly.
forged_header  = base64.urlsafe_b64encode(b'{"alg":"none","typ":"JWT"}').rstrip(b'=').decode()
forged_payload = base64.urlsafe_b64encode(
    json.dumps({'upn':'attacker@evil.com','roles':['GlobalAdmin']}).encode()
).rstrip(b'=').decode()
forged = f'{forged_header}.{forged_payload}.not-a-real-signature'

print('❌ Naive app (no signature check):')
print('   sees user =', bad_who_is_this(forged)['upn'])
print('   sees roles =', bad_who_is_this(forged)['roles'], ' ← totally fake!')

# ✅ GOOD: fetch the IdP's public keys from JWKS and verify the RS256 signature
jwks = httpx.get(f'{ENTRA}/discovery/v2.0/keys').json()

def good_who_is_this(token: str, audience: str) -> dict:
    return jose_jwt.decode(
        token,
        jwks,
        algorithms=['RS256'],
        audience=audience,
        issuer='http://localhost:9100/contoso/v2.0',
    )

print('\n✅ Correct app (signature + issuer + audience verified):')
try:
    good_who_is_this(forged, 'api://api-b')
    raise AssertionError('the forged token was ACCEPTED — signature verification is broken')
except JWTError as e:
    print('   forged token rejected:', e)

claims = good_who_is_this(token, 'api://api-b')
assert claims['upn'] == 'alice@contoso.com', f'unexpected subject: {claims.get("upn")}'
print('   real token accepted for', claims['upn'], 'issued by', claims['iss'])

---
## Federation

**Federation** establishes trust between two identity systems so users from one can access resources in the other *without creating new accounts*.

Real-world example: Contoso (your company) partners with Fabrikam. Instead of creating Fabrikam accounts in your Entra tenant, you set up a **federated trust**. Fabrikam users authenticate with *their* IdP, and your apps trust the resulting token.

```
Fabrikam user → Fabrikam IdP: "authenticate me"
Fabrikam IdP → token (signed by Fabrikam)
Fabrikam user → Contoso app: "here's my Fabrikam token"
Contoso app → Contoso IdP: "is this token trustworthy?"
Contoso IdP: "yes, we trust Fabrikam's signing keys" → access granted
```

In Microsoft's world, **Entra External ID (B2B)** enables federation with other Entra tenants, Google, Facebook, or any SAML/OIDC IdP.

### Exam tip

Federation = trust relationship between IdPs. The user authenticates with their *home* IdP. No password syncing needed.

---
### Federation vs synchronisation vs cloud-only

Three different answers to *"where does this account live, and who checks the password?"* —
and the exam mixes them up on purpose.

| Model | Where the account object lives | Who verifies the credential | Give-away in a question |
|-------|-------------------------------|-----------------------------|-------------------------|
| **Cloud-only** | Entra ID only | Entra ID | "no on-premises directory", a brand-new tenant |
| **Synchronised (hybrid)** | On-prem AD, copied into Entra ID by Entra Connect / Cloud Sync | Entra ID (password hash sync) *or* an on-prem agent (pass-through auth) | "one username and password for both" |
| **Federated** | Still in Entra ID (synced, or a B2B guest), but flagged as federated | A **different** IdP — AD FS, another Entra tenant, Google, a SAML IdP | "authenticated by their own organisation", "trust relationship" |

Two traps worth memorising:

- Synchronisation **copies objects** between directories. Federation **copies nothing** — it
  delegates the act of authenticating to somebody else's IdP.
- Password hash sync does **not** sync the password, and not even the on-prem hash. It syncs a
  hash *of* the on-prem password hash, which cannot be replayed against on-prem AD.

---
## Summary

| Concept | Key fact |
|---------|----------|
| **Authentication** | Proves *who* you are (401 if it fails) |
| **Authorization** | Determines *what* you can do (403 if it fails) |
| **Identity Provider** | Central system that authenticates users and issues tokens |
| **Directory service** | Stores users, groups, devices in a hierarchy |
| **AD DS** | On-prem, LDAP/Kerberos, you manage domain controllers |
| **Entra ID** | Cloud, OAuth2/OIDC, managed by Microsoft |
| **Federation** | Trust between IdPs — the user authenticates at their *home* IdP |
| **Sync vs federation** | Sync copies objects; federation delegates authentication |
| **Four pillars** | Administration, authentication, authorization, auditing |

---
## Self-check — identity fundamentals

Edit `MY_ANSWERS`, re-run, and read the explanations for anything you missed.

In [ ]:
QUIZ = [
    {'id': 'Q1',
     'q': 'An API receives a token it can verify, belonging to a known user, and returns 403. What failed?',
     'options': {'A': 'Authentication', 'B': 'Authorization',
                 'C': 'Federation', 'D': 'Token signature validation'},
     'a': 'B',
     'why': 'The identity was established (that is authentication, and it succeeded). 403 means the '
            'proven identity is not permitted to do this. A failed authentication returns 401 — which is '
            'also what a *missing* token returns, because then the caller is simply unknown.'},
    {'id': 'Q2',
     'q': 'Which protocol adds an identity layer on top of OAuth 2.0 so the app learns WHO the user is?',
     'options': {'A': 'SAML 2.0', 'B': 'Kerberos', 'C': 'OpenID Connect', 'D': 'LDAP'},
     'a': 'C',
     'why': 'OIDC = OAuth 2.0 + an id_token. OAuth 2.0 on its own is an authorization framework: an '
            'access token says what the bearer may call, not who they are. SAML also does sign-in, but '
            'as XML assertions and not as a layer on OAuth.'},
    {'id': 'Q3',
     'q': 'Which service is on-premises, speaks LDAP and Kerberos, and requires you to run domain controllers?',
     'options': {'A': 'Microsoft Entra ID', 'B': 'Active Directory Domain Services (AD DS)',
                 'C': 'Microsoft Entra Domain Services', 'D': 'Microsoft Entra Connect'},
     'a': 'B',
     'why': 'AD DS is the classic on-prem directory you operate yourself. Entra ID is cloud and speaks '
            'OAuth2/OIDC/SAML — it is not "AD in the cloud". Entra Domain Services is managed '
            'LDAP/Kerberos in Azure so legacy apps can lift and shift without you running DCs.'},
    {'id': 'Q4',
     'q': 'Fabrikam staff sign in with their own Fabrikam credentials to reach a Contoso app. Contoso '
          'creates no passwords for them. What is this?',
     'options': {'A': 'Password hash synchronisation', 'B': 'Pass-through authentication',
                 'C': 'Federation', 'D': 'Cloud-only identity'},
     'a': 'C',
     'why': 'Federation is a trust relationship between identity systems: the user authenticates at '
            'their home IdP and the relying party trusts the resulting token. The two sync options are '
            'about one organisation copying its own accounts to the cloud.'},
    {'id': 'Q5',
     'q': 'An app reads the JWT payload with base64 decoding and trusts the roles claim it finds. What is wrong?',
     'options': {'A': 'Nothing — the payload is encrypted',
                 'B': 'The payload is only encoded, not signed-checked, so anyone can forge claims',
                 'C': 'It should read the header instead',
                 'D': 'JWTs cannot carry roles'},
     'a': 'B',
     'why': 'A JWT payload is base64url-encoded plaintext — readable and writable by anyone. Only '
            'verifying the signature against the IdP public keys (JWKS), plus the issuer and audience, '
            'makes the claims trustworthy.'},
    {'id': 'Q6',
     'q': 'Sign-in logs, audit logs and access reviews serve which pillar of an identity infrastructure?',
     'options': {'A': 'Administration', 'B': 'Authentication', 'C': 'Authorization', 'D': 'Auditing'},
     'a': 'D',
     'why': 'Auditing answers "who did what, when, and can we prove it". Administration is the creation '
            'and governance of the identities themselves.'},
]

MY_ANSWERS = {'Q1': 'B', 'Q2': 'C', 'Q3': 'B', 'Q4': 'C', 'Q5': 'B', 'Q6': 'D'}

score = 0
for q in QUIZ:
    mine = MY_ANSWERS.get(q['id'], '').strip().upper()
    ok = mine == q['a']
    score += ok
    print(f'{"PASS" if ok else "FAIL"}  {q["id"]}: {q["q"]}')
    for k, v in q['options'].items():
        mark = '<-- correct' if k == q['a'] else ''
        print(f'         {k}. {v} {mark}')
    print(f'         your answer: {mine or "(blank)"}')
    print(f'         why: {q["why"]}\n')
print(f'Score: {score}/{len(QUIZ)}')

assert {q['id'] for q in QUIZ} == set(MY_ANSWERS), 'every question needs an answer key entry'
assert all(q['a'] in q['options'] for q in QUIZ), 'an answer key points at an option that does not exist'


**Next**: [Notebook 2 — Entra ID and Authentication](02_entra_id_and_authentication.ipynb)